In [1]:
# ============================================
# 🧰 Install libraries
# ============================================
!pip install pandas ftfy matplotlib fasttext --quiet
!pip install pytablewriter best_download lm_dataformat --quiet
!pip install datasketch --quiet
!pip install justext --quiet
!pip install pycld2 --quiet

import os, sys, re, html, json, urllib.request
import justext
import pandas as pd
import matplotlib.pyplot as plt
from ftfy import fix_text
from collections import OrderedDict

# --- FastText compatibility patch for NumPy 2.0 ---
import numpy as np

_old_array = np.array

def _safe_array(obj, *args, **kwargs):
    # Remove old/unsupported NumPy args that FastText might pass
    kwargs.pop("copy", None)
    kwargs.pop("subok", None)
    return np.asarray(obj, *args, **kwargs)

np.array = _safe_array

import fasttext
from datasketch import MinHash, MinHashLSH
import pycld2 as cld2

# ============================================
# 🧰 Clone The Pile repo (if not already there)
# ============================================
if not os.path.isdir("/content/the-pile"):
    !git clone https://github.com/EleutherAI/the-pile.git /content/the-pile

sys.path.append("/content/the-pile")

from the_pile.utils import strip_markdown_colons, remove_advertisement

# ============================================
# 1️⃣ Load data and watermark reference
# ============================================
data_path = "/content/final_uniform_replace.jsonl"
mytext_path = "/content/myText.txt"

records = [json.loads(l) for l in open(data_path, "r", encoding="utf-8")]
df = pd.DataFrame(records)
df = df[df["is_watermarked"] == True].copy()
raw_texts = df["watermarked"].tolist()
print(f"✅ Loaded {len(raw_texts)} watermarked samples.")

# =============== 13-gram MinHashLSH document-level deduplication ===============

def get_char_shingles(text, k=13):
    """Return a set of character-level k-gram shingles."""
    text = text.replace("\n", " ")
    if len(text) <= k:
        return {text}
    return {text[i:i+k] for i in range(len(text) - k + 1)}

def minhash_for_text(text, num_perm=128, k=13):
    """Build a MinHash signature from 13-gram character shingles."""
    shingles = get_char_shingles(text, k=k)
    m = MinHash(num_perm=num_perm)
    for sh in shingles:
        m.update(sh.encode("utf-8"))
    return m

def dedupe_documents_minhash(docs, threshold=0.5, num_perm=128, k=13):
    """
    Document-level near-duplicate removal using MinHashLSH.
    Returns:
      kept_docs: list of (orig_idx, text) for unique docs
      dup_map:   list of (dup_idx, kept_key) for duplicates (optional, for stats)
    """
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    kept_docs = []
    dup_map = []

    for idx, doc in enumerate(docs):
        m = minhash_for_text(doc, num_perm=num_perm, k=k)
        # query for near-duplicates among already-inserted docs
        matches = lsh.query(m)
        if not matches:
            # new unique document
            key = f"doc_{idx}"
            lsh.insert(key, m)
            kept_docs.append((idx, doc))  # keep original index for later alignment
        else:
            # near-duplicate of some existing doc
            dup_map.append((idx, matches[0]))  # (duplicate_idx, representative_key)

    return kept_docs, dup_map

# Run MinHashLSH deduplication on the raw texts
kept_docs, dup_map = dedupe_documents_minhash(raw_texts, threshold=0.5, num_perm=128, k=13)

print(f"🧹 After MinHashLSH dedupe: {len(kept_docs)} unique documents (from {len(raw_texts)})")
if dup_map:
    print(f"ℹ️ Removed {len(dup_map)} near-duplicate documents.")

# Replace `texts` with deduplicated texts but keep original indices
dedup_indices  = [i for (i, _) in kept_docs]   # indices into raw_texts
texts          = [t for (_, t) in kept_docs]   # deduplicated texts

# --- Load invisible characters from myText.txt ---
with open(mytext_path, "r", encoding="utf-8") as f:
    chars = f.read()
ZWC = [c for c in chars if not c.isprintable() and c != "\n"]
ZWC_CODES = [f"U+{ord(c):04X}" for c in ZWC]
print(f"💧 Loaded {len(ZWC)} invisible watermark characters:")
print(", ".join(ZWC_CODES[:10]) + (" ..." if len(ZWC_CODES) > 10 else ""))

# ============================================
# 2️⃣ Cleaning functions (The Pile-level + your own)
# ============================================

def clean_html(raw_html):
    """
    Clean HTML using jusText (boilerplate removal only),
    output is a single text string without paragraph restructuring.
    """
    try:
        # Run jusText to detect boilerplate
        paragraphs = justext.justext(raw_html, justext.get_stoplist("English"))

        # Keep ONLY non-boilerplate text
        cleaned_parts = [
            para.text for para in paragraphs
            if not para.is_boilerplate
        ]

        # Flatten into single string, preserve minimal spacing
        cleaned = " ".join(cleaned_parts)
        cleaned = html.unescape(cleaned).strip()

        return cleaned

    except Exception:
        # In case jusText crashes on malformed HTML
        return html.unescape(raw_html).strip()

def fix_unicode(text):
    return fix_text(text)

def remove_duplicate_lines(text):
    lines = text.splitlines()
    seen = OrderedDict()
    for line in lines:
        line = line.strip()
        if line and line not in seen:
            seen[line] = True
    return "\n".join(seen.keys())

def is_english_pycld2(text, min_chars=50):
    """
    Use pycld2 to keep only reliably detected English text.
    """
    snippet = text.strip()
    if len(snippet) < min_chars:
        return False, "too short for language detection"

    try:
        reliable, _, details = cld2.detect(snippet, bestEffort=True)
        # details[0]: (language name, language code, percent, score)
        lang_code = details[0][1]  # e.g. 'EN'
        if reliable and lang_code.lower() == "en":
            return True, f"pycld2: reliable English ({lang_code})"
        else:
            return False, f"pycld2: non-English or unreliable (reliable={reliable}, lang={lang_code})"
    except Exception as e:
        return False, f"pycld2 error: {e}"

# ============================================
# 3️⃣ Load FastText model (for stats only)
# ============================================
ft_path = "lid.176.bin"

def ensure_fasttext_model(path):
    if os.path.exists(path):
        if os.path.getsize(path) < 10 * 1024 * 1024:
            print("⚠️ Existing lid.176.bin looks too small, removing and re-downloading...")
            os.remove(path)

    if not os.path.exists(path):
        print("📥 Downloading FastText model (lid.176.bin)...")
        urllib.request.urlretrieve(
            "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin",
            path,
        )
        print("✅ Downloaded FastText model.")

ensure_fasttext_model(ft_path)
ft_model = fasttext.load_model(ft_path)

fasttext_stats = []

# ============================================
# 4️⃣ Cleaning with pycld2 English filter + FastText (stats) + debug
# ============================================
def pile_clean_text_debug(text, min_quality=0.5):
    # 1) The Pile-style light cleaning
    text = strip_markdown_colons(text)
    text = remove_advertisement(text)

    # 2) HTML → plain text via jusText
    cleaned_html = clean_html(text)
    if not cleaned_html.strip():
        return None, "HTML removal -> empty"

    # 3) Unicode fix + intra-document line dedupe
    text = fix_unicode(cleaned_html)
    text = remove_duplicate_lines(text)

    # 4) length
    if len(text.strip()) < 50:
        return None, "too short (<50 chars)"

    # 5) pycld2
    is_en, lang_reason = is_english_pycld2(text)
    if not is_en:
        return None, f"filtered by pycld2 — {lang_reason}"

    # 6) FastText
    try:
        labels, probs = ft_model.predict(text.replace("\n", " "), k=1)
        label = labels[0].replace("__label__", "")
        prob = float(probs[0])
    except Exception as e:
        label, prob = "error", 0.0

    fasttext_stats.append((label, prob))

    return text, None

# ============================================
# 5️⃣ Apply cleaning (with index tracking)
# ============================================
cleaned_records = []
filtered = []

for j, t in enumerate(texts):
    orig_idx = dedup_indices[j]  # index into raw_texts / ZWC
    cleaned, reason = pile_clean_text_debug(t)
    if cleaned:
        cleaned_records.append((orig_idx, cleaned))
    else:
        filtered.append((orig_idx, reason))

print(f"\n✅ Cleaned {len(cleaned_records)} / {len(texts)}")
for i, reason in filtered[:10]:
    print(f"🚫 Sample {i} filtered out — {reason}")

if fasttext_stats:
    labels_only = [lbl for (lbl, p) in fasttext_stats]
    label_counts = pd.Series(labels_only).value_counts()
    print("\n🌐 FastText language stats (top 10):")
    print(label_counts.head(10))

# ============================================
# 6️⃣ Per-document (1-to-1) watermark retention analysis — index aligned
# ============================================
def count_char(text, ch):
    return text.count(ch)

results = []
for orig_idx, cleaned_text in cleaned_records:
    if orig_idx >= len(ZWC):
        break
    wm_char = ZWC[orig_idx]
    orig_text = raw_texts[orig_idx]  # use original (pre-dedup, pre-cleaning) text

    orig_count = count_char(orig_text, wm_char)
    cleaned_count = count_char(cleaned_text, wm_char)
    retention_ratio = (cleaned_count / orig_count) if orig_count > 0 else 0
    reduced = cleaned_count < orig_count
    results.append({
        "doc_id": orig_idx,
        "char_code": f"U+{ord(wm_char):04X}",
        "orig_count": orig_count,
        "cleaned_count": cleaned_count,
        "retention_ratio": retention_ratio,
        "reduced": reduced
    })

df_ret = pd.DataFrame(results)

if df_ret.empty:
    print("\n⚠️ No documents passed cleaning — nothing to analyze for watermark retention.")
else:
    retained_docs = (df_ret["cleaned_count"] > 0).sum()
    reduced_docs = df_ret["reduced"].sum()
    overall_retention = (
        df_ret["cleaned_count"].sum() / df_ret["orig_count"].sum() * 100
        if df_ret["orig_count"].sum() > 0 else 0
    )

    print(f"\n✅ Per-document (1-to-1) watermark analysis complete (index-aligned).")
    print(f"💧 Documents with retained watermark: {retained_docs}/{len(df_ret)}")
    print(f"📉 Watermark reduced in {reduced_docs} documents.")
    print(f"✅ Overall watermark retention rate: {overall_retention:.2f}%")

    print("\n📊 Sample of per-document watermark retention:")
    print(df_ret.head(10))

    # ============================================
    # 7️⃣ Visualization — per-document retention distribution
    # ============================================
    plt.figure(figsize=(6,4))
    plt.hist(df_ret["retention_ratio"] * 100, bins=10, color="#4CAF50", edgecolor="black")
    plt.xlabel("Per-document watermark retention (%)")
    plt.ylabel("Document count")
    plt.title("Per-document (1-to-1) Watermark Retention after Cleaning")
    plt.grid(alpha=0.3)
    plt.show()
